In [ ]:
# Imports & paths

import sys, os
sys.path.insert(0, os.path.abspath(".."))
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
 
PROCESSED = "../data/processed"
FIGURES   = "../results/figures"
os.makedirs(FIGURES, exist_ok=True)
 
TICKERS = ["SPY", "QQQ", "GLD", "VEQT"]
COLORS  = {"SPY": "#378ADD", "QQQ": "#7F77DD", "GLD": "#EF9F27", "VEQT": "#1D9E75"}
 
prices  = pd.read_csv(f"{PROCESSED}/prices.csv",  index_col=0, parse_dates=True)
returns = pd.read_csv(f"{PROCESSED}/returns.csv", index_col=0, parse_dates=True)
 
print("Prices shape:",  prices.shape)
print("Returns shape:", returns.shape)
print(prices.tail(3))

In [ ]:
# Normalised price history (base 100)

fig, ax = plt.subplots(figsize=(11, 4))
 
for t in TICKERS:
    norm = prices[t] / prices[t].iloc[0] * 100
    ax.plot(norm.index, norm, label=t, color=COLORS[t], linewidth=1.4)
 
ax.set_title("Normalised price history (base = 100)", fontsize=13)
ax.set_ylabel("Indexed price")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.legend()
ax.grid(axis="y", linewidth=0.4, color="#cccccc")
fig.tight_layout()
fig.savefig(f"{FIGURES}/price_history.png", dpi=150)
plt.show()
print("Saved → results/figures/price_history.png")

In [ ]:
# Daily return distributions

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()
 
for i, t in enumerate(TICKERS):
    ax = axes[i]
    data = returns[t].dropna()
    ax.hist(data, bins=80, color=COLORS[t], alpha=0.85, edgecolor="none")
    ax.axvline(data.mean(), color="#444", linewidth=1, linestyle="--", label=f"mean={data.mean():.4f}")
    ax.set_title(f"{t} daily returns", fontsize=11)
    ax.set_xlabel("Daily return")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=9)
    ax.grid(axis="y", linewidth=0.3, color="#cccccc")
 
fig.suptitle("Daily return distributions (2015–2024)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(f"{FIGURES}/return_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/figures/return_distributions.png")

In [ ]:
# Rolling 20-day volatility


rolling_vol = returns.rolling(20).std() * np.sqrt(252)   # annualised
 
fig, ax = plt.subplots(figsize=(11, 4))
for t in TICKERS:
    ax.plot(rolling_vol.index, rolling_vol[t], label=t, color=COLORS[t], linewidth=1.2)
 
ax.set_title("Rolling 20-day annualised volatility", fontsize=13)
ax.set_ylabel("Annualised volatility")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.legend()
ax.grid(axis="y", linewidth=0.4, color="#cccccc")
fig.tight_layout()
fig.savefig(f"{FIGURES}/rolling_volatility.png", dpi=150)
plt.show()
print("Saved → results/figures/rolling_volatility.png")

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5))
corr = returns.corr()
 
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, ax=ax, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, linewidths=0.5, square=True,
    cbar_kws={"shrink": 0.8}
)
ax.set_title("ETF return correlation matrix", fontsize=12)
fig.tight_layout()
fig.savefig(f"{FIGURES}/correlation_heatmap.png", dpi=150)
plt.show()
print("Saved → results/figures/correlation_heatmap.png")

In [ ]:
# Descriptive statistics table

stats = returns.describe().T
stats["skew"]     = returns.skew()
stats["kurtosis"] = returns.kurtosis()
stats = stats[["mean", "std", "min", "25%", "50%", "75%", "max", "skew", "kurtosis"]]
stats.columns = ["Mean", "Std", "Min", "25%", "50%", "75%", "Max", "Skew", "Kurtosis"]
print("\nDescriptive statistics (daily returns):\n")
print(stats.round(5).to_string())
stats.round(5).to_csv(f"{PROCESSED}/../results/metrics/eda_stats.csv")
print("\nSaved → results/metrics/eda_stats.csv")

In [ ]:
# Cumulative return comparison

cum_returns = (1 + returns).cumprod()
 
fig, ax = plt.subplots(figsize=(11, 4))
for t in TICKERS:
    ax.plot(cum_returns.index, cum_returns[t], label=t, color=COLORS[t], linewidth=1.4)
 
ax.set_title("Cumulative return (2015–2024)", fontsize=13)
ax.set_ylabel("Portfolio value (starting at 1.0)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.legend()
ax.grid(axis="y", linewidth=0.4, color="#cccccc")
fig.tight_layout()
fig.savefig(f"{FIGURES}/cumulative_returns.png", dpi=150)
plt.show()
print("Saved → results/figures/cumulative_returns.png")